# China GDP Estimation

**Tabular Regression Project** — Estimate China's GDP growth curve.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: China GDP (55 rows × 2 columns)
Target: `Value` (GDP)

## 2 · Project Overview

We estimate **China's GDP** over time using a tiny dataset of just 55 rows (Year, GDP Value). The primary challenge is extracting enough features from a single time column to make meaningful predictions.

This project demonstrates regression on extremely small data and introduces the concept of growth curve modeling.

## 3 · Learning Objectives

1. Work with an extremely small dataset (55 rows).
2. Engineer temporal and polynomial features from a single Year column.
3. Understand growth curve patterns in economic data.
4. Compare linear and non-linear models on trend data.
5. Use LazyPredict and FLAML for benchmarking.

## 4 · Problem Statement

Given **Year**, predict China's **GDP Value**. This teaches trend modeling and feature engineering from minimal data.

## 5 · Why This Project Matters

- GDP estimation is a core macroeconomic task.
- China's exponential growth curve is a famous pattern in economic data.
- Working with tiny datasets is a common real-world constraint.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 55 |
| **Columns** | 2 (Year, Value) |
| **Target** | Value (GDP) |
| **File** | `china_gdp.csv` (local) |

## 7 · Dataset Source and License Notes

- **Source**: Public economic data (World Bank / IMF style).
- **License**: Public domain.
- **Limitations**: Only 55 data points spanning ~55 years.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'Value'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
DATA_PATH = os.path.join(SAVE_DIR, 'china_gdp.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing:', df.isnull().sum().sum())
print(f'Year range: {df["Year"].min()} to {df["Year"].max()}')
print(f'GDP range: [{df[TARGET].min():.2f}, {df[TARGET].max():.2f}]')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df['Year'], df[TARGET], 'bo-')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('GDP')
axes[0].set_title('China GDP Over Time')

axes[1].plot(df['Year'], np.log1p(df[TARGET]), 'ro-')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('log(1 + GDP)')
axes[1].set_title('Log-transformed GDP')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')

## 15 · Train / Validation / Test Split

Random 80/20 split. For proper time-series forecasting a temporal split would be needed.

## 16 · Preprocessing / 17 · Feature Engineering

We engineer polynomial and temporal features from the Year column to capture exponential growth.

In [ ]:
# Feature engineering
df['year_norm'] = df['Year'] - df['Year'].min()
df['year_sq'] = df['year_norm'] ** 2
df['year_cube'] = df['year_norm'] ** 3
df['year_log'] = np.log1p(df['year_norm'])
df['decade'] = (df['Year'] // 10) * 10
df['decade_enc'] = (df['decade'] - df['decade'].min()) // 10

feature_cols = ['Year', 'year_norm', 'year_sq', 'year_cube', 'year_log', 'decade_enc']
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)
print(f'Baseline LR: RMSE={baseline_rmse:.2f}, MAE={baseline_mae:.2f}, R²={baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor
lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=200, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- China's GDP follows an exponential growth curve — polynomial features capture this well.
- The `year_cube` feature is critical for modeling the acceleration in recent decades.
- Linear regression with polynomial features can compete with tree models on trend data.
- GDP estimation helps policymakers and investors assess economic trajectories.

## 26 · Limitations

- Only 55 data points — extremely limited.
- No external features (population, inflation, trade balance).
- Random split doesn't respect temporal ordering.
- Growth curves can change — past trends don't guarantee future patterns.

## 27 · How to Improve This Project

1. Add macroeconomic features (population, inflation, trade).
2. Use proper time-based train/test split.
3. Try sigmoid/logistic growth curve fitting.
4. Compare with Prophet or ARIMA.
5. Add GDP data for other countries for context.

## 28 · Production Considerations

- GDP forecasting requires many more inputs.
- Structural breaks (policy changes, crises) invalidate growth curves.
- Uncertainty quantification is essential.
- Regular model updates needed.

## 29 · Common Mistakes

1. Using only Year as feature — too simple for non-linear growth.
2. Extrapolating growth curves far into the future.
3. Random split on time data.
4. Not log-transforming exponential data.

## 30 · Mini Challenge / Exercises

1. Fit a sigmoid growth curve using scipy.optimize.
2. Use temporal split (train on first 45 years, test on last 10).
3. Add per-capita GDP as a secondary target.
4. Compare polynomial degree 2 vs. 3 vs. 5.

## 31 · Final Summary / Key Takeaways

- Polynomial features are essential for modeling exponential growth.
- On 55 rows, simpler models can match or beat complex ones.
- GDP estimation teaches growth curve analysis — a key economic ML skill.
- Feature engineering from a single column is the main challenge.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['baseline'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')